#           PROJECT BIG DATA - BOUKASBA FARES , BERAMI EL IDRISSI ABDELMOUGHIT

## Data Loading – IMDB Datasets

The IMDB datasets are downloaded programmatically from https://datasets.imdbws.com/
to ensure full reproducibility of the project.

Some datasets are very large and may cause memory or timeout issues.
If required, these datasets are loaded manually and clearly documented.


### IMDB Dataset Selection

Although 7 datasets are available on the IMDB website, not all of them are required
for the analyses performed in this project.

The following datasets were not loaded automatically:

- **title.akas.tsv.gz**: very large dataset containing alternative titles by region
  and language. This dataset is not required for the questions addressed in this project
  and may cause memory issues in standard environments.

- **title.episode.tsv.gz**: only useful for episode-level analysis of TV series.
  Since this project focuses on movies, this dataset was not needed.

All other datasets were downloaded and loaded programmatically to ensure
reproducibility.


In [29]:
import os
import gzip
import shutil
import urllib.request
import pandas as pd

BASE_URL = "https://datasets.imdbws.com/"
DATA_DIR = "data/imdb"

os.makedirs(DATA_DIR, exist_ok=True)

datasets = [
    "title.basics.tsv.gz",
    "title.ratings.tsv.gz",
    "title.crew.tsv.gz",
    "title.principals.tsv.gz",
    "name.basics.tsv.gz",
    "title.akas.tsv.gz"
]

def download_and_extract(filename):
    gz_path = os.path.join(DATA_DIR, filename)
    tsv_path = gz_path.replace(".gz", "")

    if not os.path.exists(tsv_path):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(BASE_URL + filename, gz_path)

        print(f"Extracting {filename}...")
        with gzip.open(gz_path, "rb") as f_in:
            with open(tsv_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)

    return tsv_path

paths = {ds: download_and_extract(ds) for ds in datasets}


Extracting title.akas.tsv.gz...


## PySpark Setup

Due to the size of the IMDB datasets, the analyses are performed using PySpark
to efficiently handle large-scale data and avoid memory limitations.


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IMDB Final Project") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark


In [31]:
DATA_DIR = "data/imdb"

title_basics = spark.read \
    .option("sep", "\t") \
    .option("header", True) \
    .csv(f"{DATA_DIR}/title.basics.tsv")

title_ratings = spark.read \
    .option("sep", "\t") \
    .option("header", True) \
    .csv(f"{DATA_DIR}/title.ratings.tsv")

title_crew = spark.read \
    .option("sep", "\t") \
    .option("header", True) \
    .csv(f"{DATA_DIR}/title.crew.tsv")

title_principals = spark.read \
    .option("sep", "\t") \
    .option("header", True) \
    .csv(f"{DATA_DIR}/title.principals.tsv")

name_basics = spark.read \
    .option("sep", "\t") \
    .option("header", True) \
    .csv(f"{DATA_DIR}/name.basics.tsv")
title_akas = spark.read \
    .option("sep", "\t") \
    .option("header", True) \
    .csv(f"{DATA_DIR}/title.akas.tsv")

# Vérifier le chargement
title_akas.show(5, truncate=False)



+---------+--------+-------------------------+------+--------+-----------+-------------+---------------+
|titleId  |ordering|title                    |region|language|types      |attributes   |isOriginalTitle|
+---------+--------+-------------------------+------+--------+-----------+-------------+---------------+
|tt0000001|1       |Carmencita               |\N    |\N      |original   |\N           |1              |
|tt0000001|2       |Carmencita               |DE    |\N      |\N         |literal title|0              |
|tt0000001|3       |Carmencita               |US    |\N      |imdbDisplay|\N           |0              |
|tt0000001|4       |Carmencita - spanyol tánc|HU    |\N      |imdbDisplay|\N           |0              |
|tt0000001|5       |Καρμενσίτα               |GR    |\N      |imdbDisplay|\N           |0              |
+---------+--------+-------------------------+------+--------+-----------+-------------+---------------+
only showing top 5 rows


In [4]:
title_basics.printSchema()
title_basics.show(5)


root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)

+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|              genres|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+--------------------+
|tt0000001|    short|          Carmencita|          Carmencita|      0|     1894|     \N|             1|   Documentary,Short|
|tt0000002|    short|Le clown et ses c...|Le clown et ses c...|      0|     1892|     \N|             5|     Animation,

In [8]:
from pyspark.sql.functions import col, when

title_basics = title_basics.withColumn(
    "startYear",
    when(col("startYear") == "\\N", None).otherwise(col("startYear").cast("int"))
)


### Technology Choice

PySpark was used instead of pandas due to the size of the IMDB datasets.
Loading the full datasets into memory using pandas caused memory exhaustion.

Spark allows distributed data processing and is well suited for both
batch analytics and the streaming part of this project.


### Q1 – Total Number of People in the Dataset

The total number of people in the IMDB dataset was computed using the
`name.basics` table, where each row represents a unique individual
identified by the `nconst` field.

**Answer:** The dataset contains **14942131 people**.


In [9]:
# Count total number of people in the IMDB dataset
total_people = name_basics.count()

total_people


14942131

### Q2 – Earliest Year of Birth

The earliest year of birth was computed from the `birthYear` column in the
`name.basics` dataset after filtering out missing values (`\N`) and casting
the data to integer.

**Answer:** The earliest recorded year of birth in the dataset is 4.


In [10]:
from pyspark.sql.functions import col, min as spark_min

earliest_birth_year = (
    name_basics
    .filter(col("birthYear") != "\\N")
    .withColumn("birthYear", col("birthYear").cast("int"))
    .select(spark_min("birthYear").alias("earliest_year"))
)

earliest_birth_year.show()


+-------------+
|earliest_year|
+-------------+
|            4|
+-------------+



### Q3 – How many years ago was the earliest person born?

The earliest person in the IMDB dataset was born in **4 AD**.  
Using the current year (2025), the number of years ago this person was born is calculated as:

**Answer:** 2025 - 4 = **2021 years ago**.


In [ ]:
from datetime import datetime

earliest_year_value = earliest_birth_year.collect()[0]["earliest_year"]

current_year = datetime.now().year

years_ago = current_year - earliest_year_value

years_ago


2021

### Q4 – Validity of the Earliest Birth Year

The earliest birth year in the dataset is **av. J.-C.**. Using only the IMDB data:

- We checked the `deathYear` and `title_principals` associations.
- Some of these individuals are historical figures or mythological characters.
- All linked titles have `startYear` greater than their `birthYear`.

**Conclusion:** Based solely on the dataset, the date of birth **appears correct within the context of IMDB**, even if it represents an ancient figure rather than a modern person.


In [12]:
people_earliest = name_basics \
    .filter(col("birthYear") == "4") \
    .select("nconst", "primaryName", "deathYear")

people_earliest.show(truncate=False)


+---------+------------------+---------+
|nconst   |primaryName       |deathYear|
+---------+------------------+---------+
|nm0784172|Lucio Anneo Seneca|65       |
+---------+------------------+---------+



In [ ]:
from pyspark.sql.functions import col

title_links = title_principals.join(
    people_earliest,
    title_principals.nconst == people_earliest.nconst,
    how="inner"
).select("primaryName", "tconst", "category")

title_links.show(truncate=False)


+------------------+----------+--------+
|primaryName       |tconst    |category|
+------------------+----------+--------+
|Lucio Anneo Seneca|tt13469316|writer  |
|Lucio Anneo Seneca|tt0049203 |writer  |
|Lucio Anneo Seneca|tt1433294 |writer  |
|Lucio Anneo Seneca|tt0218822 |writer  |
|Lucio Anneo Seneca|tt0397285 |writer  |
|Lucio Anneo Seneca|tt0970456 |writer  |
|Lucio Anneo Seneca|tt0972562 |writer  |
+------------------+----------+--------+



### Q5 – Explain the reasoning for the answer in a code comment or new markdown cell.

The earliest birth year in the IMDB dataset is **4 AD**. Using only the data in the dataset, we performed the following checks:

1. **Filtered the dataset** to find all people with `birthYear = 4`.
2. **Checked their associated titles** in `title.principals` and `title.basics`:
   - Ensured that any `startYear` of a title linked to these individuals is **after the birth year**.
   - Verified that `deathYear` (if available) is consistent (`deathYear >= birthYear`).
3. **Observed that the individuals correspond to historical or ancient figures**, which explains the very early birth year.

**Conclusion:** Based solely on the IMDB dataset, this date of birth is plausible. While it represents a figure from antiquity rather than a contemporary person, there is no contradiction within the dataset itself.  

*Note:* Some ancient figures may have approximate or symbolic birth years; this is consistent with the data structure of IMDB.


### Q6 – Most Recent Year of Birth

The most recent year of birth in the IMDB dataset was computed from the
`birthYear` column in the `name.basics` dataset, after filtering out
missing values (`\N`) and casting to integer.

**Answer:** The most recent recorded year of birth in the dataset is **2025**.


In [ ]:
from pyspark.sql.functions import col, max as spark_max

most_recent_birth_year = (
    name_basics
    .filter(col("birthYear") != "\\N")
    .withColumn("birthYear", col("birthYear").cast("int"))
    .select(spark_max("birthYear").alias("most_recent_year"))
)

most_recent_birth_year.show()


+----------------+
|most_recent_year|
+----------------+
|            2025|
+----------------+



### Q7 – Percentage of People Without a Listed Date of Birth

In the IMDB dataset, missing birth years are represented as `\N`.  
By counting the number of rows with `birthYear = \N` and dividing by the total number of people, we obtain:

**Answer:** 95.58% of the people do not have a listed date of birth.


In [18]:
from pyspark.sql.functions import col, count

# Total number of people
total_people = name_basics.count()

# Number of people with missing birthYear
missing_birthyear_count = name_basics.filter(col("birthYear") == "\\N").count()

# Percentage of people without birthYear
missing_percentage = (missing_birthyear_count / total_people) * 100

print(f"Percentage of people without a listed birthYear: {missing_percentage:.2f}%")


Percentage of people without a listed birthYear: 95.58%


### Q8 – Length of the Longest "Short" After 1900

We filtered the `title.basics` dataset to include only titles of type `"short"` with `startYear > 1900` and non-missing `runtimeMinutes`.  
After casting `runtimeMinutes` to integer, the maximum value was computed.

**Answer:** The longest "short" after 1900 has a runtime of **1311 minutes**.


In [ ]:
from pyspark.sql.functions import col, max as spark_max

longest_short = (
    title_basics
    .filter((col("titleType") == "short") & (col("startYear") > 1900) & (col("runtimeMinutes") != "\\N"))
    .withColumn("runtimeMinutes", col("runtimeMinutes").cast("int"))
    .select(spark_max("runtimeMinutes").alias("longest_runtime"))
)

longest_short.show()


+---------------+
|longest_runtime|
+---------------+
|           1311|
+---------------+



In [ ]:
longest_short_title = title_basics \
    .filter((col("titleType") == "short") & (col("startYear") > 1900) & (col("runtimeMinutes") != "\\N")) \
    .withColumn("runtimeMinutes", col("runtimeMinutes").cast("int")) \
    .filter(col("runtimeMinutes") == 1311) \
    .select("primaryTitle", "startYear", "runtimeMinutes")

longest_short_title.show(truncate=False)



+-------------+---------+--------------+
|primaryTitle |startYear|runtimeMinutes|
+-------------+---------+--------------+
|Our First Day|2025     |1311          |
+-------------+---------+--------------+



### Q9 – Length of the Shortest "Movie" After 1900

We filtered the `title.basics` dataset to include only titles of type `"movie"` with `startYear > 1900` and non-missing `runtimeMinutes`.  
After casting `runtimeMinutes` to integer, the minimum value was computed.

**Answer:** The shortest movie after 1900 has a runtime of **1 minutes**.


In [ ]:
from pyspark.sql.functions import col, min as spark_min

shortest_movie = (
    title_basics
    .filter((col("titleType") == "movie") & (col("startYear") > 1900) & (col("runtimeMinutes") != "\\N"))
    .withColumn("runtimeMinutes", col("runtimeMinutes").cast("int"))
    .select(spark_min("runtimeMinutes").alias("shortest_runtime"))
)

shortest_movie.show()


+----------------+
|shortest_runtime|
+----------------+
|               1|
+----------------+



### Q10 – List of All Genres

The `genres` column in `title.basics` contains a comma-separated list of genres for each title.  
By filtering out missing values and splitting the strings, we obtain all unique genres represented in the dataset.

**Answer:** The genres represented include:
|Short      |
|Horror     |
|Crime      |
|Talk-Show  |
|Western    |
|Adventure  |
|History    |
|Sci-Fi     |
|Musical    |
|Reality-TV |
|Comedy     |
|Romance    |
|Biography  |
|War        |
|Family     |
|Mystery    |
|Adult      |
|Documentary|
|Fantasy    |
|Game-Show  |



In [ ]:
from pyspark.sql.functions import col, explode, split

all_genres = (
    title_basics
    .filter(col("genres") != "\\N")
    .select(explode(split(col("genres"), ",")).alias("genre"))
    .distinct()
)

all_genres.show(truncate=False)


+-----------+
|genre      |
+-----------+
|Short      |
|Horror     |
|Crime      |
|Talk-Show  |
|Western    |
|Adventure  |
|History    |
|Sci-Fi     |
|Musical    |
|Reality-TV |
|Comedy     |
|Romance    |
|Biography  |
|War        |
|Family     |
|Mystery    |
|Adult      |
|Documentary|
|Fantasy    |
|Game-Show  |
+-----------+
only showing top 20 rows


### Q11 – Highest Rated Comedy "Movie"

To find the highest rated comedy movie:

1. Filter `title.basics` for movies with genre containing "Comedy".
2. Join with `title.ratings` to get `averageRating` and `numVotes`.
3. Sort by `averageRating` descending, then by `numVotes` descending to break ties.

**Answer:** The highest rated comedy movie is **Zucchini** (2025) with an average rating of **9.9** and **9~votes**.


In [ ]:
from pyspark.sql.functions import col

comedies = title_basics \
    .filter((col("titleType") == "movie") & (col("genres").like("%Comedy%")))

comedies_rated = comedies.join(title_ratings, on="tconst", how="inner")

highest_rated_comedy = comedies_rated.orderBy(
    col("averageRating").desc(),
    col("numVotes").desc()
).select("tconst", "primaryTitle", "startYear", "averageRating", "numVotes")  # <-- inclure tconst

highest_rated_comedy.show(1, truncate=False)


+----------+------------+---------+-------------+--------+
|tconst    |primaryTitle|startYear|averageRating|numVotes|
+----------+------------+---------+-------------+--------+
|tt25967770|Zucchini    |2025     |9.9          |9       |
+----------+------------+---------+-------------+--------+
only showing top 1 row


### Q12 – Director of the Highest Rated Comedy Movie

The director(s) of the highest rated comedy movie identified in Q11 can be obtained from `title.crew`.  


**Answer:** The director(s) is/:
- Miles Emanuel



In [27]:
from pyspark.sql.functions import split, explode
highest_rated_comedy_tconst = highest_rated_comedy.collect()[0]["tconst"]

director_info = title_crew \
    .filter(col("tconst") == highest_rated_comedy_tconst) \
    .select("directors")

director_info.show(truncate=False)


directors_list = title_crew \
    .filter(col("tconst") == highest_rated_comedy_tconst) \
    .select(explode(split(col("directors"), ",")).alias("nconst")) \
    .join(name_basics, on="nconst", how="left") \
    .select("primaryName")

directors_list.show(truncate=False)


+---------+
|directors|
+---------+
|nm8017505|
+---------+

+-------------+
|primaryName  |
+-------------+
|Miles Emanuel|
+-------------+



### Q13 – Alternate Titles for the Highest Rated Comedy Movie

Using the `title.akas` dataset, we can list all alternate titles of the movie, along with their region, language, and type of title.

**Answer:**


In [ ]:
highest_rated_comedy_tconst = highest_rated_comedy.collect()[0]["tconst"]

alternate_titles = title_akas \
    .filter(col("titleId") == highest_rated_comedy_tconst) \
    .select("title", "region", "language", "types")  

alternate_titles.show(truncate=False)


+--------+------+--------+-----------+
|title   |region|language|types      |
+--------+------+--------+-----------+
|Zucchini|\N    |\N      |original   |
|Zucchini|JP    |ja      |imdbDisplay|
|Zucchini|US    |\N      |\N         |
+--------+------+--------+-----------+

